# Deploy Trained Policy

<img src="./media/rollout.gif" width="480" height="360">

Deploy trained policy in simulation.

In [22]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
import numpy as np
from lerobot.datasets.image_writer import AsyncImageWriter
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.configs.types import FeatureType
from lerobot.datasets.factory import resolve_delta_timestamps
from lerobot.datasets.utils import dataset_to_policy_features
import torch
from PIL import Image
import torchvision
from pathlib import Path

## Load Policy

In [11]:
device = 'cuda'

In [13]:
dataset_metadata = LeRobotDatasetMetadata("nov_19_e20", root='./nov_19_e20')
features = dataset_to_policy_features(dataset_metadata.features)
output_features = {key: ft for key, ft in features.items() if ft.type is FeatureType.ACTION}
input_features = {key: ft for key, ft in features.items() if key not in output_features}
input_features.pop("observation.wrist_image")
# Policies are initialized with a configuration class, in this case `DiffusionConfig`. For this example,
# we'll just use the defaults and so no arguments other than input/output features need to be passed.
# Temporal ensemble to make smoother trajectory predictions
cfg = ACTConfig(input_features=input_features, output_features=output_features, chunk_size= 10, n_action_steps=1, temporal_ensemble_coeff = 0.9)
delta_timestamps = resolve_delta_timestamps(cfg, dataset_metadata)
# We can now instantiate our policy with this config and the dataset stats.
policy = ACTPolicy.from_pretrained('./ckpt/act_y', config = cfg)
policy.to(device)

Loading weights from local directory


ACTPolicy(
  (model): ACT(
    (vae_encoder): ACTEncoder(
      (layers): ModuleList(
        (0-3): 4 x ACTEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
          )
          (linear1): Linear(in_features=512, out_features=3200, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=3200, out_features=512, bias=True)
          (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
      (norm): Identity()
    )
    (vae_encoder_cls_embed): Embedding(1, 512)
    (vae_encoder_robot_state_input_proj): Linear(in_features=6, out_features=512, bias=True)
    (vae_encoder_action_input_proj): Linear(in_features=8, out_features=512, bias

## Load Environment

In [19]:
from mujoco_env.papras7dof_env import PaprasEnv
xml_path = './asset/papras_scene.xml'
PnPEnv = PaprasEnv(xml_path, action_type='joint_angle')


-----------------------------------------------------------------------------
name:[Tabletop] dt:[0.002] HZ:[500]
 n_qpos:[39] n_qvel:[35] n_qacc:[35] n_ctrl:[11]
 integrator:[RK4]

n_body:[31]
 [0/31] [world] mass:[0.00]kg
 [1/31] [front_object_table] mass:[1.00]kg
 [2/31] [camera] mass:[0.00]kg
 [3/31] [camera2] mass:[0.00]kg
 [4/31] [camera3] mass:[0.00]kg
 [5/31] [robot1/link1] mass:[0.86]kg
 [6/31] [robot1/link2] mass:[0.95]kg
 [7/31] [robot1/link3] mass:[0.50]kg
 [8/31] [robot1/link4] mass:[0.60]kg
 [9/31] [robot1/link5] mass:[1.16]kg
 [10/31] [robot1/link6] mass:[0.45]kg
 [11/31] [robot1/link7] mass:[0.43]kg
 [12/31] [robot1/end_link] mass:[0.02]kg
 [13/31] [robot1/wrist_link] mass:[0.00]kg
 [14/31] [robot1/gripper_main_link] mass:[0.24]kg
 [15/31] [robot1/gripper_link] mass:[0.07]kg
 [16/31] [robot1/gripper_link_r2] mass:[0.02]kg
 [17/31] [robot1/gripper_link_l1] mass:[0.07]kg
 [18/31] [robot1/gripper_link_l2] mass:[0.02]kg
 [19/31] [robot1/end_effector_link] mass:[0.00]kg
 [2

## Roll-Out Your Policy

In [86]:
step = 0
time_out = 10 #seconds
PnPEnv.reset(seed=None)
policy.reset()
policy.eval()
save_image = True
img_transform = torchvision.transforms.ToTensor()
episodes = 10
cur_episode = 0
record = True
shape = None
record_folder = "./11_19/act_5000steps"
image_writer = AsyncImageWriter(num_threads = 4)
while PnPEnv.env.is_viewer_alive():
    PnPEnv.step_env()
    if PnPEnv.env.loop_every(HZ=20):
        # Check if the task is completed
        success = PnPEnv.check_success()
        if success or step > time_out * 20:
            print('Success')
            # Reset the environment and action queue
            policy.reset()
            PnPEnv.reset(seed=None)
            step = 0
            save_image = False
            cur_episode += 1
            if record and cur_episode >= episodes:
                break
        # Get the current state of the environment
        state = PnPEnv.get_ee_pose()
        # Get the current image from the environment
        image, wirst_image = PnPEnv.grab_image()
        if not shape:
            shape = image.shape
        image = Image.fromarray(image)
        if record:
            fpath = Path(record_folder + f"/{cur_episode}" + f"/{step}.jpg")
            fpath.parent.mkdir(parents = True, exist_ok = True)
            image_writer.save_image(image, Path(record_folder + f"/{cur_episode}" + f"/{step}.jpg"))
        image = image.resize((256, 256))
        image = img_transform(image)
        wrist_image = Image.fromarray(wirst_image)
        wrist_image = wrist_image.resize((256, 256))
        wrist_image = img_transform(wrist_image)
        data = {
            'observation.state': torch.tensor([state]).to(device),
            'observation.image': image.unsqueeze(0).to(device),
            'observation.wrist_image': wrist_image.unsqueeze(0).to(device),
            'task': ['Put mug cup on the plate'],
            'timestamp': torch.tensor([step/20]).to(device)
        }
        # Select an action
        action = policy.select_action(data)
        action = action[0].cpu().detach().numpy()
        # Take a step in the environment
        _ = PnPEnv.step(action)
        PnPEnv.render()
        step += 1
        success = PnPEnv.check_success()
        if success:
            print('Success')
            break

DONE INITIALIZATION
Success
DONE INITIALIZATION
Success
DONE INITIALIZATION
Success
DONE INITIALIZATION
Success
DONE INITIALIZATION
Success
DONE INITIALIZATION
Success
DONE INITIALIZATION
Success
DONE INITIALIZATION
Success
DONE INITIALIZATION
Success
DONE INITIALIZATION
Success
DONE INITIALIZATION


In [57]:
shape

(600, 800, 3)

In [87]:
import shutil
import cv2
for folder in Path(record_folder).iterdir():
    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Codec for .mp4
    video_path = Path("./videos/")
    video_path.mkdir(exist_ok = True, parents=True)
    ep = str(folder).split("/")[-1]
    video = cv2.VideoWriter(video_path / Path(f"{ep}.mp4"), fourcc, 20, shape[:2])
    for file in sorted(folder.iterdir(), key= lambda x: int(str(x).split("/")[-1][:-4])):
        img = cv2.imread(file)
        img = cv2.resize(img, shape[:2]) 
        if img is not None:
            # print(img.shape)
            video.write(img)
    video.release()
    print(folder, ep)
    shutil.rmtree(folder)

11_19/act_5000steps/9 9
11_19/act_5000steps/8 8
11_19/act_5000steps/6 6
11_19/act_5000steps/4 4
11_19/act_5000steps/3 3
11_19/act_5000steps/7 7
11_19/act_5000steps/1 1
11_19/act_5000steps/2 2
11_19/act_5000steps/5 5
11_19/act_5000steps/0 0
